# ORC: Создание таблицы и запись через INSERT

## 1. Инициализация локального Spark

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("ORC Insert Demo") \
    .config("spark.sql.warehouse.dir", "./spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/29 20:43:06 WARN Utils: Your hostname, MacBook-Air-Pavel.local, resolves to a loopback address: 127.0.0.1; using 192.168.50.18 instead (on interface en0)
26/03/29 20:43:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 20:43:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

## 2. DDL — создание таблицы в формате ORC

In [4]:
spark.sql("""
    DROP TABLE IF EXISTS employees
""")

spark.sql("""
    CREATE TABLE employees (
        id      INT,
        name    STRING,
        dept    STRING,
        salary  DOUBLE
    )
    STORED AS ORC
""")

print("Таблица создана")

26/03/29 20:43:13 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/03/29 20:43:13 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore pavel@127.0.0.1
26/03/29 20:43:13 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException
26/03/29 20:43:13 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


Таблица создана


26/03/29 20:43:13 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/03/29 20:43:13 WARN HiveMetaStore: Location: file:/Users/pavel/Documents/github/DE/Advanced Spark/spark-warehouse/employees specified for non-external table:employees


## 3. INSERT — запись данных в ORC

In [5]:
spark.sql("""
    INSERT INTO employees VALUES
        (1, 'Иван Петров',    'Аналитика',  90000.0),
        (2, 'Мария Иванова',  'Инженерия',  110000.0),
        (3, 'Алексей Сидоров','Аналитика',  95000.0),
        (4, 'Елена Козлова',  'Менеджмент', 120000.0),
        (5, 'Дмитрий Новиков','Инженерия',  105000.0)
""")

print("Данные записаны")

Данные записаны


## 4. Чтение и проверка

In [6]:
df = spark.sql("SELECT * FROM employees ORDER BY id")
df.show()

+---+---------------+----------+--------+
| id|           name|      dept|  salary|
+---+---------------+----------+--------+
|  1|    Иван Петров| Аналитика| 90000.0|
|  2|  Мария Иванова| Инженерия|110000.0|
|  3|Алексей Сидоров| Аналитика| 95000.0|
|  4|  Елена Козлова|Менеджмент|120000.0|
|  5|Дмитрий Новиков| Инженерия|105000.0|
+---+---------------+----------+--------+



In [7]:
# Проверяем физический путь к ORC файлам
import os

warehouse = "./spark-warehouse/employees"
for f in os.listdir(warehouse):
    path = os.path.join(warehouse, f)
    size = os.path.getsize(path)
    print(f"{f}  —  {size} байт")

part-00004-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc  —  830 байт
.part-00000-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc.crc  —  16 байт
part-00003-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc  —  839 байт
._SUCCESS.crc  —  8 байт
part-00000-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc  —  823 байт
.part-00001-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc.crc  —  16 байт
.part-00004-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc.crc  —  16 байт
.part-00002-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc.crc  —  16 байт
_SUCCESS  —  0 байт
part-00002-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc  —  843 байт
part-00001-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc  —  817 байт
.part-00003-b988d0c5-060f-47f4-b1e8-020390722b8b-c000.zstd.orc.crc  —  16 байт


In [8]:
spark.stop()